# Olist Data Cleaning & Data Wrangling

## Import pandas

In [41]:
import pandas as pd

## Import files 

In [77]:
orders_df = pd.read_csv('olist_orders_dataset.csv')
order_items_df = pd.read_csv('olist_order_items_dataset.csv')
order_reviews_df = pd.read_csv('olist_order_reviews_dataset.csv')
payments_df = pd.read_csv('olist_order_payments_dataset.csv')
products_df = pd.read_csv('olist_products_dataset.csv')
product_dictionary_df = pd.read_csv('product_category_name_translation.csv')

## Basic exploration of files

### 1. Orders dataset

In [43]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


We can already identify an issue. The data type of the date columns need to be changed to date time for any comparisons to be done correctly. So let's fix that. 

In [44]:
orders_df[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date']] = orders_df[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date']].apply(pd.to_datetime)
orders_df[['order_delivered_customer_date', 'order_estimated_delivery_date']] = orders_df[['order_delivered_customer_date', 'order_estimated_delivery_date']].apply(pd.to_datetime)

orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


## Data types have been changed
![Noice Gif](https://media1.tenor.com/m/H6sjheSkU1wAAAAd/noice-nice.gif)

Lets continue looking at our orders table.

In [45]:
#Get statuses of different orders
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

From here, we can see different order statuses. A large portion of orders have the "delivered" status. Other statuses give us some insight into the dataset and what had happened to other orders. 

What is intriguing is the *"unavailable"* status. What could that possibly mean? We will have to decipher it through some exploration of the data.

### (1.1) Understanding null values

In [46]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

The order null values start at the "order_approved_at" column. Time to examine the statuses of these deliveries and understand the context behind these null values.

#### (i) Order approved at

In [47]:
orders_df.loc[orders_df['order_approved_at'].isnull(), 'order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

The orders which were "canceled" makes sense (it means that the orders were cancelled pre-payment approval), alongside the "created" orders (pressumably means that the order was created but payment has not been approved). 

The puzzling thing here is the *"delivered"* status, because **how can an order be fulfilled, yet payment seems to not have been approved?**. We need to dive deeper. 

In [48]:
#Storing it in a seperate dataframe in case of extra operations on this filtered dataset needs to be done
non_approved_delivered_df = orders_df.loc[(orders_df['order_approved_at'].isnull()) & (orders_df['order_status'] == 'delivered')]
non_approved_delivered_df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


The orders did not have a payment approval date, but all the other timestamps have data. The orders were supposedly "fulfilled." We will have to reconcile these orders with the payments table. 

It is a bit unusual but **lets check the next columns.**

#### (ii) Order delivered carrier date

In [49]:
orders_df.loc[orders_df['order_delivered_carrier_date'].isnull(), 'order_status'].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

The "unavailable" order status has made an appearance. This means that 609 orders are not visible within the system for some reason. They will have to be dealt with prior to any table joins. 

The number of "cancelled" orders has increased (509). This means the orders were cancelled prior to the order being handed over to the carrier.
The "invoiced" and "processing" orders are interesting. The payments have been approved (the statuses did not show up when the approval date was null), so these statuses are on the end of Olist and/or the carrier. 

The following be can be assumed:

*"Processing" orders - Orders that are being packaged and will make way to the carrier.*

*"Invoiced" orders - Orders where the package is **either** en route to the carrier or yet to be processed.*

The "delivered" orders are puzzling. The value decreased from the previous column. Let's do some investigation. 

In [50]:
orders_df.loc[(orders_df['order_delivered_carrier_date'].isnull()) & (orders_df['order_status'] == 'delivered')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


Looking at this, we see:

(i) There 2 orders, both have been approved. 

(ii) One order **has a customer delivery date, but no carrier delivery date**, and the other **has no customer or carrier delivery date**.

The former order will have to be checked with the payments table, and the latter will probably be removed in any operations for sales and revenue.

In the other columns

The new order statuses have reached the total count of their status (Unavailable = 609; Created = 5; Approved = 2). We'll have to monitor these values for any changes.

*I'm aware that we'll probably remove most of these rows (for fulfillment and revenue analysis) once tables are joined, however, I need to determine the status of orders for analysis of operations pertaining to logistics*

#### (iii) Order delivered customer date

In [51]:
orders_df.loc[orders_df['order_delivered_customer_date'].isnull(), 'order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

New order status "shipped" indicates to us that orders have reached the carrier but are yet to be delivered. 

The following statuses have not changed values: unavailable, invoiced, processing, created, approved.

The number of "cancelled" shipments have increased - meaning they were cancelled after arriving at the carrier, but before being delivered. 

"Delivered" orders have changed values once more. At this point, I must change my career to Detective, with all of the investigations happening.

In [52]:
orders_df.loc[(orders_df['order_delivered_customer_date'].isnull()) & (orders_df['order_status'] == 'delivered')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


Looking at this, we see:

(i) There 8 orders. 

(ii) 7 orders **have a carrier delivery date, but no customer delivery date**, and 1 order **has no customer or carrier delivery date**.

Same process, check payments table and remove a record.

**The last column has no null values**

### (1.2) Checking for duplicates

We have to verify that the data has no duplicates. We'll be using the "order_id" column for verification. It is the *only* truly unique value in each row.

In [53]:
orders_df.duplicated('order_id').value_counts()

False    99441
Name: count, dtype: int64

No duplicate rows in the table. That's good :)

### (1.3) Ensuring data integrity

Now that we have checked for duplicates and understood the context behind the null values, we need to verify that the data is trustworthy in terms of data validity and integrity. 

We cannot assume that all the dates of the table align. It is enitrely possible that "delivery" dates are **somehow before the date of purchase or approval**. We need to check to ensure that the data is valid.

#### (i) Order purchase vs other columns

In [54]:
print('Comparison (purch vs approve): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_approved_at']].shape[0], '# of incorrect rows')
print('Comparison (purch vs deliv_carrier): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_delivered_carrier_date']].shape[0], '# of incorrect rows')
print('Comparison (purch vs deliv_customer): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_delivered_customer_date']].shape[0], '# of incorrect rows')
print('Comparison (purch vs deliv_estimate): ', orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_estimated_delivery_date']].shape[0], '# of incorrect rows')

Comparison (purch vs approve):  0 # of incorrect rows
Comparison (purch vs deliv_carrier):  166 # of incorrect rows
Comparison (purch vs deliv_customer):  0 # of incorrect rows
Comparison (purch vs deliv_estimate):  0 # of incorrect rows


There are 166 rows of purchase dates being later than the date of delivery to the carrier. Let's see the data visually.

In [60]:
 orders_df.loc[orders_df['order_purchase_timestamp'] > orders_df['order_delivered_carrier_date']].head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,e6f5b234bb0d847f10eebd70130c5d49,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,72fcbb1145f2889eddcba6d6c1d6c090,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,ccd8e3459ad58ae538c9d2cf35532ba4,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,5c427f06e7fcf902e309743516e2c580,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
2760,c3eb293fd154223498b6551a728203e8,1701ed88b1e8cb21051660d70d11d038,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36,2018-08-06
3473,b0c2a7d04b165525254254a728c50a4e,76a8a3092e48bba0826495241903e450,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43,2018-07-04
3661,2033a4586b5bec3229ebc1675a8ae092,0efc1f5e8675901683fae0ea8fdf3feb,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27,2018-07-19
4114,08adcddad19d3acf37d1fa01cb9ded1e,94c2f45091985c47b4c4fb301113f0a4,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53,2018-07-18
4159,dee6298ce7d1fb2645141ef9972157aa,c35fd04a09c83d6f53b2d4557eab7445,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaT,2018-05-28


Upon closer inspection, we see that some of the dates align but the times of the carrier delivery date compared to the purchase time are off.

Let's check other columns.

#### (ii) Order payment approval vs other columns

In [61]:
print('Comparison (approve vs deliv_carrier): ', orders_df.loc[orders_df['order_approved_at'] > orders_df['order_delivered_carrier_date']].shape[0], '# of incorrect rows')
print('Comparison (approve vs deliv_customer): ', orders_df.loc[orders_df['order_approved_at'] > orders_df['order_delivered_customer_date']].shape[0], '# of incorrect rows')
print('Comparison (approve vs deliv_estimate): ', orders_df.loc[orders_df['order_approved_at'] > orders_df['order_estimated_delivery_date']].shape[0], '# of incorrect rows')

Comparison (approve vs deliv_carrier):  1359 # of incorrect rows
Comparison (approve vs deliv_customer):  61 # of incorrect rows
Comparison (approve vs deliv_estimate):  12 # of incorrect rows


In [62]:
orders_df.loc[orders_df['order_approved_at'] > orders_df['order_delivered_carrier_date']].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06


#### (iii) Order Carrier Date vs other columns

In [63]:
print('Comparison (deliv_carrier vs deliv_customer): ', orders_df.loc[orders_df['order_delivered_carrier_date'] > orders_df['order_delivered_customer_date']].shape[0], '# of incorrect rows')
print('Comparison (deliv_carrier vs deliv_estimate): ', orders_df.loc[orders_df['order_delivered_carrier_date'] > orders_df['order_estimated_delivery_date']].shape[0], '# of incorrect rows')

Comparison (deliv_carrier vs deliv_customer):  23 # of incorrect rows
Comparison (deliv_carrier vs deliv_estimate):  473 # of incorrect rows


### (1.3.1) Date fixing method

When researching common methods (with the help of AI) for date inconsistency fixing, I came across a few methods:

(a) Using imputation: Adding a calculated number to fix the date OR to use another date (forward fill) to change the incorrect date 
(b) Strict Nullification : Changing the incorrect date to a null type, so as to not add bias to the data.
(c) Row removal: Removing the row where the incorrect date resides (very dangerous for further analysis)

I chose to go with option B, due to the need for correctness of data for analysis and predictive modelling. Python (Pandas specifically) ignores null values when calculating means and medians.

I will use the 'order_approved_at' column as a basis for comparison for the following reason(s):

1. The process of order fulfillment begins when the payment has been approved (presumably).
2. More date errors reside in the comparisons with this column against others.

Lets "fix" the dates.

#### (i) Orders approved vs order carrier delivered date

In [88]:
orders_df = pd.read_csv('olist_orders_dataset.csv')

In [89]:
approve_carrier_mask = (orders_df['order_approved_at'] > orders_df['order_delivered_carrier_date']) & (orders_df['order_status'] == 'delivered')
orders_df.loc[approve_carrier_mask, 'order_delivered_carrier_date'] = pd.NaT
orders_df.loc[approve_carrier_mask]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,NaN,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,NaN,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,NaN,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,NaN,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,NaN,2018-07-28 00:05:39,2018-08-06 00:00:00
...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,NaN,2018-07-10 23:21:47,2018-07-24 00:00:00
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,NaN,2018-07-10 17:47:39,2018-07-23 00:00:00
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,NaN,2018-03-18 20:08:50,2018-03-02 00:00:00
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,NaN,2018-04-28 17:38:42,2018-05-15 00:00:00


#### (ii) Orders approved vs order customer delivered date

In [ ]:
orders_df = pd.read_csv('olist_orders_dataset.csv')

In [90]:
approve_customer_mask = (orders_df['order_approved_at'] > orders_df['order_delivered_customer_date']) & (orders_df['order_status'] == 'delivered')
orders_df.loc[approve_customer_mask, 'order_delivered_customer_date'] = pd.NaT
orders_df.loc[approve_customer_mask]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,NaN,NaN,2018-07-31 00:00:00
483,4df92d82d79c3b52c7138679fa9b07fc,ba0660bf3fffe505ee892e153a2fbd49,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,NaN,NaN,2018-08-06 00:00:00
1986,6e57e23ecac1ae881286657694444267,2dda54e25d0984e12705c84d4030e6e0,delivered,2018-08-09 17:36:47,2018-08-20 15:55:42,NaN,NaN,2018-09-06 00:00:00
3659,f222c56f035b47dfa1e069a88235d730,b74ca180d63f9ae0443e4e13a2f5bdaf,delivered,2018-01-30 09:43:45,2018-02-04 23:31:47,NaN,NaN,2018-02-15 00:00:00
11738,cf72398d0690f841271b695bbfda82d2,2b7fff075bda701552485ef3f0810257,delivered,2017-09-01 18:45:33,2017-09-13 22:04:39,NaN,NaN,2017-09-21 00:00:00
...,...,...,...,...,...,...,...,...
89703,fcbf4f4ef049367f9f85af94ed3b6010,814d433efda6ac4b0859b5a71aac64c8,delivered,2018-04-20 15:11:15,2018-04-24 18:41:20,NaN,NaN,2018-04-30 00:00:00
91801,4387477eec4b3c89b39f3f454940d059,dbbbdb6ab9a7d8e3fc39a907f456bf19,delivered,2018-08-09 20:45:10,2018-08-20 15:56:29,NaN,NaN,2018-08-14 00:00:00
93680,4f3a6e28d764cf896b1fceb0028422c8,e24fceba7ac4b1b1cd1884f05ba68e9d,delivered,2018-07-03 09:34:16,2018-07-05 16:21:50,NaN,NaN,2018-07-20 00:00:00
94359,9c3186381b733d4304e2e416afc6bbc1,a4bd2fc7bf4fc0061011ef71a2208d63,delivered,2018-07-28 20:49:05,2018-08-02 23:30:29,NaN,NaN,2018-08-03 00:00:00


#### (iii) Orders approved vs order estimated delivered date

In [ ]:
orders_df = pd.read_csv('olist_orders_dataset.csv')

In [91]:
approve_estimate_mask = (orders_df['order_approved_at'] > orders_df['order_estimated_delivery_date']) & (orders_df['order_status'] == 'delivered')
orders_df.loc[approve_estimate_mask, 'order_estimated_delivery_date'] = pd.NaT
orders_df.loc[approve_estimate_mask]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
43418,5e981569f5835c96e4b288363b3b8f63,53bb5604662705879bbe3f11a25bda0f,delivered,2018-07-30 16:25:37,2018-08-02 13:43:13,NaN,NaN,NaN
50623,e73fe43cdcd166f7f0c6e3c2bf11a917,3a412111e4f293539eae05b502fc63de,delivered,2018-08-09 18:06:43,2018-08-20 15:57:28,NaN,NaN,NaN
55353,9675440ebf61a1a3482cc6308e3ebd28,9597de20c8b55d0f22dcc26fdaa6b2f1,delivered,2018-08-18 23:35:23,2018-08-24 22:05:08,2018-08-27 16:14:00,2018-08-28 17:37:30,NaN
57637,394d17c2b71a726e205caaeee3d2aa3d,10d109b1f1f45bd2bf9afd83543eedd4,delivered,2018-08-03 14:56:26,2018-08-08 23:31:22,NaN,NaN,NaN
61544,6dcf0aeb8b1eb4021c26e1d0e9394979,8d10ccfe7c979ef982e4964bbff62637,delivered,2018-08-09 20:37:34,2018-08-20 15:59:18,NaN,NaN,NaN
91801,4387477eec4b3c89b39f3f454940d059,dbbbdb6ab9a7d8e3fc39a907f456bf19,delivered,2018-08-09 20:45:10,2018-08-20 15:56:29,NaN,NaN,NaN


In [93]:
orders_df.loc[orders_df['order_status'] == 'delivered'].isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date     1352
order_delivered_customer_date      69
order_estimated_delivery_date       6
dtype: int64

The null values have increased across the board, but to assure that the remaining data is accurate, it needed to be done.

### 2. Order Items dataset

In [99]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [123]:
order_items_df.head(20)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


### 3. Order Reviews dataset

### 4. Order Payments dataset

### 5. Products dataset

## Dataframe manipulation

In [124]:
grouped_order_items_df = order_items_df.groupby('order_id').agg({
    'order_item_id': 'count',       # Total items per order
    'price': 'sum',                 # Total price per order
    'freight_value': 'sum'          # Total freight per order
}).reset_index()
grouped_order_items_df.columns = ['order_id', 'total_items', 'total_price', 'total_freight']
grouped_order_items_df[grouped_order_items_df['total_items'] > 1].head(10)

,order_id,total_items,total_price,total_freight
13,0008288aa423d2a3f00fcb17cd7d8719,2,99.80,26.74
31,00143d0f86d6fbd9f9b38ab440ac16f5,3,63.99,45.30
39,001ab0a7578dd66cd4b0a71f5b6e1e41,3,74.67,52.89
43,001d8f0e34a38c37f7dba2a37d4eba8b,2,37.98,15.56
69,002c9def9c9b951b1bec6d50753c9891,2,156.00,17.80
73,002f98c0f7efd42638ed6100ca699b42,2,53.89,39.73
79,003324c70b19a16798817b2b3640e721,2,205.80,28.90
82,00337fe25a3780b3424d9ad7c5a4b35e,2,119.80,19.88
91,003822434f91204da0a51fe4cf2aba18,2,138.00,37.16
100,003f201cdd39cdd59b6447cff2195456,2,170.00,34.06


## Combining datasets

## Cleaning dataframe & engineering features